# 13 - Phase 5: cross-dataset generalisation

**Main question**: does MiniConvNet generalise beyond the CT dataset it was trained and validated
on, or is its performance specific to that dataset?

**No retraining, no fitting, no threshold-tuning on external data.** Every model here is loaded from
an existing `.keras` checkpoint and run through **exactly one forward pass** over the external
dataset. There is no `.fit()` call, no optimizer step, and nothing about the external data is used
to choose any parameter, threshold, or temperature anywhere in this notebook or in
`src/external_eval_utils.py`.

**No new models.** Primary is MiniConvNet; secondary is VGG16 (the strongest Phase 2 fine-tuned
baseline, read programmatically from `results/fair_baseline/metrics_fair_baseline.csv` - not
hardcoded, exactly as Phases 3/4 did), included only after a cost check (Step 0c below) rather than
assumed to be "computationally practical" without evidence.

**Nothing in this notebook has been executed.** It was written, not run - the external dataset has
not been downloaded, no checkpoint has been loaded, and no evaluation has taken place. Dataset
research (below) used live web search, which is research, not execution.

---

## STEP 1 - dataset research (done now, documented here with sources)

Two candidates were evaluated, as instructed. **Both were verified against real sources, not
assumed** - see below.

### Candidate A: IQ-OTH/NCCD (Iraq-Oncology Teaching Hospital / National Center for Cancer Diseases) - SELECTED

| Property | Finding | Source |
|---|---|---|
| Modality | **Confirmed CT** - originally DICOM, Siemens SOMATOM scanner, 120 kV, 1mm slice thickness | [Mendeley Data](https://data.mendeley.com/datasets/bhmdr45bh2/4), [GitHub mirror](https://github.com/hamdalla93/The-IQ-OTHNCCD-lung-cancer-dataset) |
| Labels | **3 whole-slice-image classes: Normal / Benign / Malignant** (110 cases; image counts vary slightly by release/mirror - ~1097-1190 total slice images reported across sources) | [Kaggle (hamdallak)](https://www.kaggle.com/datasets/hamdallak/the-iqothnccd-lung-cancer-dataset), [Kaggle (waseemnagahhenes)](https://www.kaggle.com/datasets/waseemnagahhenes/lung-cancer-dataset-iq-othnccd) |
| Independence from our primary dataset | **Confirmed independent.** Collected clinically at IQ-OTH/NCCD, fall 2019, by oncologists/radiologists at that hospital. Our primary ("Chest CT-Scan images", Mohamed Hany) dataset is, by its own Kaggle description, images "hand collected from various websites" - an unrelated, web-aggregated compilation with no described clinical acquisition protocol. Different institution, different acquisition, different annotation process. | [IQ-OTH/NCCD Mendeley](https://data.mendeley.com/datasets/bhmdr45bh2/4), [primary dataset Kaggle page](https://www.kaggle.com/datasets/mohamedhanyyy/chest-ctscan-images) |
| Licence / access | **CC BY 4.0**, publicly available on Kaggle and Mendeley Data, no special credentialing (same acquisition pattern already used for the primary dataset) | [Kaggle](https://www.kaggle.com/datasets/hamdallak/the-iqothnccd-lung-cancer-dataset) |
| Format on the public mirrors | Originally DICOM; public Kaggle/Mendeley redistributions are PNG/JPG (standard practice for this kind of public mirror) - **verified in code, not assumed** (see Step 1 code cell: `scan_for_dicom()`) | inferred from redistribution pattern; confirmed by the notebook itself when run |
| 4-class compatibility | **No** - no NSCLC subtype information at all. A reduced task is required (Step 2). |  |

### Candidate B: LIDC-IDRI - REJECTED

| Property | Finding | Source |
|---|---|---|
| Modality | Confirmed CT (1018 scans, 7 medical centres) | [arXiv survey](https://arxiv.org/pdf/2003.06801) |
| Labels | **Nodule-level malignancy SCORE, 1-5, ordinal**, from up to 4 radiologists **without enforced consensus** - not a whole-slice discrete class label at all | multiple arXiv sources on LIDC-IDRI annotation methodology |
| Why rejected | Using this dataset would require: (a) DICOM series + XML nodule-annotation parsing (`pydicom`/`pylidc`, neither in `requirements.txt`), (b) nodule localisation/cropping - a fundamentally different unit of analysis than our checkpoints' whole-slice classification, (c) an arbitrary score-to-label threshold decision (is a malignancy score of 3, "uncertain", benign or malignant?). This is manufacturing a task the data was not built for, not reducing an existing one - exactly what the addendum prohibits. **No further code was written for this candidate.** |  |

### The reduced task (Step 2, in detail)

Our checkpoints output 4-class softmax probabilities over `CLASS_NAMES` (adenocarcinoma / large cell
carcinoma / normal / squamous cell carcinoma). These three tumour-subtype classes plus `normal`
**fully partition** the softmax output, so summing the three subtype probabilities gives an exact
(not renormalised, not approximated) `p_tumor`, alongside `p_normal` - see
`external_eval_utils.reduce_to_binary_tumor_probs()`.

IQ-OTH/NCCD's `benign` class has **no counterpart anywhere in our training data** - our primary
dataset has no benign category at all. Silently grouping it with either `normal` or `malignant` would
be an unjustified modelling choice. **Rather than picking one grouping silently, three separate,
explicitly labelled framings are computed and reported** (`external_eval_utils.EXTERNAL_TASK_FRAMINGS`):

| Framing | Scope | Status |
|---|---|---|
| `malignant_vs_normal_excl_benign` | benign cases **excluded** from scoring entirely | **PRIMARY** - most defensible, only scores categories with a genuine counterpart in our training data |
| `malignant_vs_not_malignant_full` | full set; benign grouped with normal | secondary/exploratory |
| `abnormal_vs_normal_full` | full set; benign grouped with malignant | secondary/exploratory - most lenient, most favourable to the model |

A fourth, purely **descriptive** (never accuracy-scored) report covers what the model actually
predicts on benign inputs specifically (`describe_benign_subset()`) - since there is no correct
answer for "benign" in our label space, nothing about that behaviour is graded right or wrong.

**Every result from this notebook must be read against this reduced task, not as a like-for-like
4-class comparison with the in-domain results.** This is restated at every results table below, not
just here.

### Acquiring the dataset (for whoever runs this)

Download from [Kaggle](https://www.kaggle.com/datasets/hamdallak/the-iqothnccd-lung-cancer-dataset)
or [Mendeley Data](https://data.mendeley.com/datasets/bhmdr45bh2) (CC BY 4.0) into
`./Data_External_IQOTHNCCD/` locally, or attach as a Kaggle input -
`external_eval_utils.resolve_external_data_root()` auto-detects either. Folder names are matched
case-insensitively against known spelling variants (the commonly-cited public mirror spells the
benign folder "Bengin case", a carried-through typo - not independently confirmed byte-for-byte
during this research since Kaggle's file listing is JS-rendered and not visible to a plain fetch, so
the resolver tries both spellings and **prints exactly which folder names it found** for the runner
to verify).

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

import time
import numpy as np
import pandas as pd

from src.config import *
from src.gradcam_utils import identify_strongest_finetuned_baseline, verify_required_checkpoints
from src.finetune_utils import CHECKPOINTS_LOCAL

from src.external_eval_utils import (
    ensure_external_eval_dirs, EXTERNAL_EVAL_DIR, EXTERNAL_FIGURES_DIR,
    EXTERNAL_DATA_DIR, EXTERNAL_CLASS_NAMES, EXTERNAL_TASK_FRAMINGS,
    resolve_external_data_root, find_external_class_dirs, scan_for_dicom,
    index_external_dataset, external_class_counts, sample_image_properties,
    make_external_dataset, run_external_forward_pass, print_external_eval_time_estimate,
    reduce_to_binary_tumor_probs, evaluate_external_framing, describe_benign_subset,
    load_in_domain_binary_reference, domain_shift_summary,
    plot_external_confusion, plot_domain_shift_bar,
    record_external_result, load_external_results,
    save_external_predictions, save_benign_report,
)
from src.calibration_utils import (
    calibration_summary, plot_reliability_diagram, plot_confidence_by_correctness,
    fit_temperature, apply_temperature, CALIBRATION_METRICS_CSV,
)

pd.set_option('display.width', 200)
print('external-eval output dirs:', ensure_external_eval_dirs())

## Step 0a - identify the secondary model programmatically (same logic as Phases 3/4)

In [ ]:
secondary = identify_strongest_finetuned_baseline()

if secondary['available']:
    print('Strongest fine-tuned baseline from Phase 2 (secondary model for this phase):')
    print(f"  model      : {secondary['model']}")
    print(f"  run_name   : {secondary['run_name']}")
    print(f"  accuracy   : {secondary['accuracy']:.4f}")
    print(f"  checkpoint : {secondary['checkpoint_path']}")
else:
    print('Could not determine a secondary model:', secondary['reason'])
    print('MiniConvNet-only external evaluation can still proceed once its checkpoint is verified.')

PRIMARY_RUN_NAME = 'miniconvnet_single_run'
PRIMARY_CHECKPOINT = str(CHECKPOINTS_LOCAL / f'{PRIMARY_RUN_NAME}.keras')
SECONDARY_RUN_NAME = secondary['run_name'] if secondary['available'] else None
SECONDARY_MODEL_NAME = secondary['model'] if secondary['available'] else None
SECONDARY_CHECKPOINT = secondary['checkpoint_path'] if secondary['available'] else None

## Step 0b - verify both checkpoints exist BEFORE anything else runs

Same gate as Phases 3/4: a missing checkpoint is reported, never trained around.

In [ ]:
ck_status = verify_required_checkpoints(
    PRIMARY_CHECKPOINT, SECONDARY_CHECKPOINT,
    secondary_label=f"{SECONDARY_MODEL_NAME or 'secondary'} (strongest Phase 2 baseline)")

print(ck_status['primary']['message'])
print(ck_status['secondary']['message'])

PRIMARY_OK = ck_status['primary']['exists']
SECONDARY_OK = ck_status['secondary']['exists']
print()
print('PRIMARY_OK  :', PRIMARY_OK)
print('SECONDARY_OK:', SECONDARY_OK)

## STEP 1 (code) - locate and verify the external dataset

Resolves the dataset, checks for raw DICOM (would block this pipeline - see `scan_for_dicom()`'s
docstring), indexes every image, and prints the actual class counts and a sample of real file
properties (format/mode/size) - the concrete "confirm it's really CT" check, run on whatever was
actually downloaded rather than trusted from documentation alone.

In [ ]:
EXTERNAL_ROOT = None
try:
    EXTERNAL_ROOT = resolve_external_data_root()
    print('external data root:', EXTERNAL_ROOT)
except FileNotFoundError as exc:
    print('External dataset not found:')
    print(' ', exc)
    print('\nDownload it (see the markdown above) before continuing - the cells below will fail '
          'without it. This is expected and not an error in the code.')

In [ ]:
external_df = None
if EXTERNAL_ROOT is not None:
    dicom_check = scan_for_dicom(EXTERNAL_ROOT)
    print(f"DICOM files found: {dicom_check['n_dicom_files']}")
    print(' ', dicom_check['note'])

    if dicom_check['is_dicom']:
        print('\nSTOPPING external-dataset indexing - raw DICOM is not handled by this module.')
    else:
        class_dirs = find_external_class_dirs(EXTERNAL_ROOT)
        print('\nfolder names actually found, per class (verify these look right):')
        for cls, dirs in class_dirs.items():
            for d in dirs:
                print(f'  {cls:10s} <- {d}')

        external_df = index_external_dataset(EXTERNAL_ROOT)
        print(f'\nimages indexed: {len(external_df)}')
        print(external_class_counts(external_df).to_string())

        print('\nsample file properties (format/mode/size) - confirms these are real images, not')
        print('placeholders or a wrong-modality mixup:')
        print(sample_image_properties(external_df).to_string(index=False))

## STEP 2 (code) - reduced-task label construction, printed explicitly

No modelling happens here - just a loud, explicit statement of the task change before any evaluation
runs, so nobody downstream can mistake this for a 4-class comparison.

In [ ]:
if external_df is not None:
    print('THE TASK HAS CHANGED FOR THIS PHASE.')
    print('=' * 78)
    print('In-domain (Phases 0-4): 4-class subtype classification')
    print(f'  {CLASS_NAMES}')
    print()
    print('This phase (external, IQ-OTH/NCCD): reduced binary task(s), against 3 external classes')
    print(f'  {EXTERNAL_CLASS_NAMES}')
    print()
    for key, framing in EXTERNAL_TASK_FRAMINGS.items():
        tag = 'PRIMARY' if framing['primary'] else 'secondary/exploratory'
        print(f"  [{tag}] {key}: {framing['label']}")
        print(f"      {framing['description']}")
    print()
    print('Plus a descriptive-only (non-scored) report of model behaviour on benign inputs.')

## STEP 0c - time estimate for the secondary model (VGG16), read before running it

Per the addendum: a single forward pass should be a small fraction of Phase 4's MC-Dropout cost
(15 passes, 265s measured for VGG16 there), but this is checked rather than assumed.

In [ ]:
RUN_SECONDARY = False
if external_df is not None and SECONDARY_OK:
    probe_ds = make_external_dataset(external_df, batch_size=32)
    import tensorflow as tf
    model_probe = tf.keras.models.load_model(SECONDARY_CHECKPOINT)
    t0 = time.time()
    for x, _y in probe_ds.take(1):
        model_probe(x, training=False)
    first_batch_seconds = time.time() - t0
    n_batches = int(np.ceil(len(external_df) / 32))
    est = print_external_eval_time_estimate(first_batch_seconds, n_batches, label=SECONDARY_MODEL_NAME)
    del model_probe
    tf.keras.backend.clear_session()

    print()
    if est['projected_total_seconds'] > 180:
        print(f"Projected {est['projected_total_seconds']:.0f}s exceeds a casual threshold for a "
              "single pass - still cheap relative to Phase 4's MC Dropout (265s for 15 passes), but")
        print('flagged here rather than silently assumed practical, per the addendum.')
    else:
        print(f"Projected {est['projected_total_seconds']:.0f}s - clearly practical for a single "
              'forward pass.')
    RUN_SECONDARY = True
elif external_df is not None:
    print(f'{SECONDARY_MODEL_NAME or "secondary model"} not available - skipping (see Step 0b).')

## STEP 3 - external evaluation: one forward pass per model, three framings + benign report

`run_external_forward_pass()` is the ONLY place a model touches the external images - everything
after it (all three framings, the benign-behaviour report, calibration) is derived from that single
cached 4-class probability array. No fitting, no threshold search, no tuning against this data
anywhere below.

In [ ]:
external_results = {}   # model_name -> {'external_label':..., 'y_prob_4class':..., 'binary_prob':...}

if external_df is not None:
    models_to_run = ['MiniConvNet'] if PRIMARY_OK else []
    if RUN_SECONDARY:
        models_to_run.append(SECONDARY_MODEL_NAME)

    for model_name in models_to_run:
        checkpoint = PRIMARY_CHECKPOINT if model_name == 'MiniConvNet' else SECONDARY_CHECKPOINT
        print(f'=== {model_name}: forward pass over {len(external_df)} external images ===')
        t0 = time.time()
        external_label, y_prob_4class, model = run_external_forward_pass(checkpoint, external_df)
        elapsed = time.time() - t0
        print(f'  wall clock: {elapsed:.1f}s')

        binary_prob = reduce_to_binary_tumor_probs(y_prob_4class)
        external_results[model_name] = {
            'external_label': external_label, 'y_prob_4class': y_prob_4class,
            'binary_prob': binary_prob, 'elapsed_seconds': elapsed,
        }
        save_external_predictions(model_name, external_df, external_label, y_prob_4class, binary_prob)

        import tensorflow as tf
        tf.keras.backend.clear_session()
else:
    print('external_df is None - dataset not available, nothing to evaluate this run.')

In [ ]:
framing_results = {}   # (model_name, framing_key) -> metrics dict
benign_reports = {}

for model_name, data in external_results.items():
    print(f'=== {model_name} ===')
    framing_results_this_model = {}
    for key in EXTERNAL_TASK_FRAMINGS:
        m = evaluate_external_framing(key, data['external_label'], data['binary_prob'])
        framing_results_this_model[key] = m
        tag = 'PRIMARY' if m['primary'] else 'secondary'
        print(f"  [{tag:9s}] {m['label']:52s} n={m['n_samples']:4d} (excluded={m['n_excluded']:4d}) "
              f"acc={m['accuracy']:.4f} f1_macro={m['f1_macro']:.4f}")
        framing_results[(model_name, key)] = m

    benign = describe_benign_subset(data['external_label'], data['binary_prob'])
    benign_reports[model_name] = benign
    if benign.get('n_benign', 0):
        print(f"  [descriptive, not scored] benign cases: n={benign['n_benign']}, "
              f"predicted-tumor fraction={benign['fraction_predicted_tumor']:.3f}, "
              f"mean confidence={benign['mean_confidence']:.3f}")
    print()

if benign_reports:
    save_benign_report(benign_reports)

In [ ]:
for (model_name, key), m in framing_results.items():
    confusion_path = plot_external_confusion(m, model_name, show=False)
    if confusion_path:
        print(f'{model_name} / {key}: {confusion_path}')

## STEP 4 - domain shift: in-domain vs external, on the PRIMARY reduced task

The in-domain reference is the already-known `binary_tumor_acc` from `outputs/results_table.csv`
(MiniConvNet's 3-fold CV row) and `tumour_detection_accuracy` from
`results/fair_baseline/metrics_fair_baseline.csv` (VGG16's Run B row) - **read live, not hardcoded**,
via `load_in_domain_binary_reference()`. This is exactly the same reduced binary task (tumour vs
normal) computed in-domain instead of externally, which is what makes this a clean before/after
comparison rather than an apples-to-oranges one.

In [ ]:
in_domain_ref = load_in_domain_binary_reference(SECONDARY_MODEL_NAME, SECONDARY_RUN_NAME)
print('in-domain reference (read live from existing results files):')
for model_name, ref in in_domain_ref.items():
    print(f"  {model_name}: {ref['binary_tumor_accuracy']:.4f}  (source: {ref['source']})")

domain_shift = {}
for model_name in external_results:
    primary_key = [k for k, f in EXTERNAL_TASK_FRAMINGS.items() if f['primary']][0]
    ext_acc = framing_results[(model_name, primary_key)]['accuracy']
    if model_name in in_domain_ref:
        ds = domain_shift_summary(in_domain_ref[model_name]['binary_tumor_accuracy'], ext_acc)
        domain_shift[model_name] = ds
        print(f'\n--- {model_name} ---')
        print(f"  in-domain accuracy (tumour vs normal): {ds['in_domain_accuracy']:.4f}")
        print(f"  external accuracy  (PRIMARY framing) : {ds['external_accuracy']:.4f}")
        print(f"  absolute drop                        : {ds['absolute_drop']:+.4f}")
        print(f"  relative drop                        : {ds['relative_drop_pct']}%")
        print(f"  {ds['direction']}")
        plot_domain_shift_bar(model_name, ds['in_domain_accuracy'], ds['external_accuracy'], show=False)
    else:
        print(f'\n--- {model_name}: no in-domain reference found, cannot compute domain shift ---')

In [ ]:
for model_name in external_results:
    for key, m in {k[1]: v for k, v in framing_results.items() if k[0] == model_name}.items():
        ds = domain_shift.get(model_name, {})
        record_external_result({
            'model': model_name, 'framing': key, 'label': m['label'], 'primary': m['primary'],
            'n_samples': m['n_samples'], 'n_excluded': m['n_excluded'],
            'accuracy': m['accuracy'], 'precision_macro': m['precision_macro'],
            'recall_macro': m['recall_macro'], 'f1_macro': m['f1_macro'],
            'precision_tumor': m['precision_tumor'], 'recall_tumor': m['recall_tumor'],
            'f1_tumor': m['f1_tumor'], 'auc': m.get('auc'), 'mean_confidence': m.get('mean_confidence'),
            'in_domain_accuracy': ds.get('in_domain_accuracy') if m['primary'] else None,
            'absolute_drop': ds.get('absolute_drop') if m['primary'] else None,
            'notes': m['description'],
        })
print('external evaluation results written to', EXTERNAL_EVAL_DIR / 'external_eval_metrics.csv')

## STEP 5 - confidence/calibration on the external set (reusing Phase 4's `calibration_utils.py`)

`calibration_summary()`, `expected_calibration_error()`, `brier_score()`, `negative_log_likelihood()`
and `plot_reliability_diagram()` are all reused directly from Phase 4 - none of them hardcode a class
count, so they apply to this phase's 2-column reduced probabilities without modification.

**Nothing is fit on the external set.** If Phase 4 already fitted a temperature on in-domain
validation data for a given model (`results/calibration/calibration_metrics.csv`), that
already-fitted temperature is optionally APPLIED here (not re-fit) to see whether an in-domain
calibration fix transfers - this is read-only reuse of a value fit entirely on in-domain data, not a
new fit against this external data.

In [ ]:
for model_name, data in external_results.items():
    primary_key = [k for k, f in EXTERNAL_TASK_FRAMINGS.items() if f['primary']][0]
    include = EXTERNAL_TASK_FRAMINGS[primary_key]['include'](data['external_label'])
    y_true_primary = EXTERNAL_TASK_FRAMINGS[primary_key]['binary_true'](data['external_label'])[include]
    y_prob_primary = data['binary_prob'][include]

    print(f'=== {model_name} (PRIMARY framing: {EXTERNAL_TASK_FRAMINGS[primary_key]["label"]}) ===')
    summary = calibration_summary(y_true_primary, y_prob_primary)
    print(f"  n_samples       : {summary['n_samples']}")
    print(f"  accuracy        : {summary['accuracy']:.4f}")
    print(f"  mean_confidence : {summary['mean_confidence']:.4f}")
    print(f"  ECE             : {summary['ece']:.4f}")
    print(f"  Brier           : {summary['brier_score']:.4f}")
    print(f"  NLL             : {summary['nll']:.4f}")
    plot_reliability_diagram(y_true_primary, y_prob_primary, model_name,
                             method_label='external_primary_framing', show=False)
    plot_confidence_by_correctness(y_true_primary, y_prob_primary, model_name,
                                   method_label='external_primary_framing', show=False)

    # Optional: apply (never fit) Phase 4's already-fitted in-domain temperature, if it exists.
    if Path(CALIBRATION_METRICS_CSV).exists():
        calib_df = pd.read_csv(CALIBRATION_METRICS_CSV)
        row = calib_df[(calib_df['model'] == model_name) & (calib_df['stage'] == 'after_temperature')]
        if len(row):
            T = float(row.iloc[0]['temperature'])
            print(f"\n  Applying Phase 4's in-domain-fitted temperature (T={T:.4f}) to the FULL "
                  '4-class external probabilities, then re-reducing to binary (NOT re-fit here):')
            y_prob_4class_scaled = apply_temperature(data['y_prob_4class'], T)
            binary_prob_scaled = reduce_to_binary_tumor_probs(y_prob_4class_scaled)
            summary_scaled = calibration_summary(y_true_primary, binary_prob_scaled[include])
            print(f"    ECE   : {summary['ece']:.4f} -> {summary_scaled['ece']:.4f}")
            print(f"    Brier : {summary['brier_score']:.4f} -> {summary_scaled['brier_score']:.4f}")
        else:
            print(f'\n  No Phase 4 temperature on record for {model_name} - skipping transfer check.')
    else:
        print('\n  Phase 4 calibration_metrics.csv not found on this machine - skipping transfer check.')
    print()

## STEP 6 - interpretation

Fill in once the cells above have actually been run. **This phase evaluates external-dataset
generalisation only - nothing here establishes or implies clinical validity.**

**1. Does MiniConvNet generalise?**
*(compare the PRIMARY framing's external accuracy against chance (0.5) and against the in-domain*
*reference from Step 4 - state plainly whether it does, partially, or does not)*

**2. How much does performance drop?**
*(the `absolute_drop` / `relative_drop_pct` values from Step 4, for each available model)*

**3. Which classes/categories are most affected?**
*(compare `precision_tumor`/`recall_tumor` against `precision_normal`/`recall_normal` from the*
*PRIMARY framing's confusion matrix; also note what the benign-subset descriptive report shows -*
*does the model treat benign findings more like tumour or more like normal?)*

**4. Does confidence remain reliable externally?**
*(compare external ECE/Brier from Step 5 against the in-domain values from*
*`results/calibration/calibration_metrics.csv` if available - report whichever direction the*
*numbers actually show)*

**5. Is there evidence the model is dataset-specific rather than learning transferable signal?**
*(a large accuracy drop combined with a shift toward chance-level or systematically biased*
*predictions would support this; a modest drop with predictions still clearly better than chance*
*would argue against it - report what was actually observed, not what would be a cleaner story)*

In [ ]:
ext_df_results = load_external_results()

print('PHASE:                Phase 5 - cross-dataset generalisation')
print('EXTERNAL DATASET:      IQ-OTH/NCCD (Iraq-Oncology Teaching Hospital / National Center for')
print('                       Cancer Diseases), CC BY 4.0, https://www.kaggle.com/datasets/hamdallak/')
print('                       the-iqothnccd-lung-cancer-dataset')
print('TASK:                  reduced binary task(s) - see the three framings above; PRIMARY =')
print('                       malignant-vs-normal with benign excluded from scoring. NOT a 4-class')
print('                       comparison with the in-domain results.')
print()

for model_name in external_results:
    primary_key = [k for k, f in EXTERNAL_TASK_FRAMINGS.items() if f['primary']][0]
    m = framing_results[(model_name, primary_key)]
    ds = domain_shift.get(model_name)
    print(f'--- {model_name} ---')
    print(f"IN-DOMAIN RESULT:      {ds['in_domain_accuracy']:.4f}" if ds else 'IN-DOMAIN RESULT:      n/a')
    print(f"EXTERNAL RESULT:       accuracy={m['accuracy']:.4f} f1_macro={m['f1_macro']:.4f} "
          f"n={m['n_samples']} (excluded={m['n_excluded']})")
    print(f"PERFORMANCE DROP:      {ds['absolute_drop']:+.4f} ({ds['relative_drop_pct']}%)" if ds
          else 'PERFORMANCE DROP:      n/a (no in-domain reference)')
    print(f"CONFIDENCE:            mean_confidence={m.get('mean_confidence', float('nan')):.4f} - "
          'see Step 5 for ECE/Brier')
    print()

print('FILES CREATED:')
print(f'  {EXTERNAL_EVAL_DIR}/external_eval_metrics.csv')
print(f'  {EXTERNAL_EVAL_DIR}/external_eval_metrics.json')
print(f'  {EXTERNAL_EVAL_DIR}/benign_subset_report.json')
print(f'  {EXTERNAL_EVAL_DIR}/predictions/  (per-model raw predictions on the external set)')
print(f'  {EXTERNAL_FIGURES_DIR}/  (confusion matrices, domain-shift bars, reliability diagrams)')
print('CPU TIME:              sum of the per-model wall-clock timings printed in Step 3 (a single')
print('                       forward pass per model - the cheapest phase in this project)')
print('LIMITATIONS:           reduced task, not 4-class (no NSCLC subtype ground truth exists')
print('                       externally); benign has no ground-truth-correct label in our scheme')
print('                       (reported descriptively only, never scored); external dataset size is')
print('                       modest (~1,100 images, 110 cases - similar order to our own ~1,000');
print('                       image primary set, so this is not a large-scale stress test); no')
print('                       claim of clinical validity is made anywhere in this notebook - this')
print('                       evaluates generalisation to one external dataset only.')
print('RESEARCH CONCLUSION:   fill in Step 6 above once run')